<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

OrderLine

### Вариант задания 
№15

<h2 style="color:DodgerBlue">Описание проекта:</h2>

Были добавлены конструкторы во все классы (OrderLine, StandardLine, SpecialLine, FreeLine) с инициализацией через геттеры и сеттеры, а не напрямую через поля — это обеспечило срабатывание валидации уже при создании объекта. Во всех свойствах была реализована проверка корректности значений (ProductId > 0, Price ≥ 0, Discount от 0 до 100, Prepayment ≥ 0 и т. д.). Также было реализовано взаимодействие объектов между собой: методы CompareWith, ApplyDiscountTo, CombineWith в базовом классе, а также AddUnitsTo (StandardLine), ShareDiscountWith (SpecialLine) и PayFor (FreeLine) — каждый из них принимает другой объект OrderLine и работает с его свойствами или вызывает его методы, изменяя его состояние.

#### Дополнительное задание
Добавьте к сущестующим классам конструктора классов с использованием гетторов и сетторов и реализуйте взаимодействие объектов между собой

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System; 

class OrderLine
{
    private int productId;
    private string productName;
    private decimal price;   // decimal — тип для денежных расчётов

    public int ProductId
    {
        get { return productId; }
        set
        {
            if (value <= 0)
                throw new ArgumentOutOfRangeException("ProductId должен быть положительным!");
            productId = value;
        }
    }

    public string ProductName
    {
        get { return productName; }
        set
        {
            if (string.IsNullOrWhiteSpace(value))
                throw new ArgumentException("Название товара не может быть пустым!");
            productName = value;
        }
    }

    public decimal Price
    {
        get { return price; }
        set
        {
            if (value < 0)
                throw new ArgumentOutOfRangeException("Цена не может быть отрицательной!");
            price = value;
        }
    }

    public OrderLine()
    {
        ProductId = 0;             // через сеттер
        ProductName = "Без названия";
        Price = 0m;
    }

    //параметризованный конструктор(инициализация через сеттеры)
    public OrderLine(int productId, string productName, decimal price)
    {
        ProductId = productId;     // через сеттер — с валидацией
        ProductName = productName;
        Price = price;
    }

    public virtual decimal CalculateTotal()
    {
        return Price;
    }

    public void UpdatePrice(decimal newPrice)
    {
        Price = newPrice;
        Console.WriteLine($"Цена обновлена: {Price:C}");
    }

    public string GetProductDetails()
    {
        return $"ID: {ProductId}, Название: {ProductName}, Цена: {Price:C}";
    }

    //смотрю взаимодействие объектов

    // Сравниваю стоимость с другим товаром
    public void CompareWith(OrderLine other)
    {
        if (other == null)
        {
            Console.WriteLine("[Сравнение] Не с чем сравнивать.");
            return;
        }
        decimal myTotal = CalculateTotal();
        decimal otherTotal = other.CalculateTotal();

        Console.WriteLine($"[Сравнение] «{ProductName}» ({myTotal:C}) vs «{other.ProductName}» ({otherTotal:C})");

        if (myTotal > otherTotal)
            Console.WriteLine($"  → «{ProductName}» дороже на {myTotal - otherTotal:C}");
        else if (myTotal < otherTotal)
            Console.WriteLine($"  → «{other.ProductName}» дороже на {otherTotal - myTotal:C}");
        else
            Console.WriteLine("  → Стоимость одинаковая.");
    }

    //Применяю скидку к другому товару
    public void ApplyDiscountTo(OrderLine other, decimal percent)
    {
        if (other == null)
        {
            Console.WriteLine("[Скидка] Некому применять скидку.");
            return;
        }
        if (percent < 0 || percent > 100)
        {
            Console.WriteLine("[Скидка] Некорректный процент скидки.");
            return;
        }
        decimal newPrice = other.Price * (1 - percent / 100);
        other.Price = newPrice;
        Console.WriteLine($"[Скидка] «{ProductName}» дарит скидку {percent}% товару «{other.ProductName}». Новая цена: {other.Price:C}");
    }

    // Объединяю два товара в один (суммарная стоимость)
    public decimal CombineWith(OrderLine other)
    {
        if (other == null)
        {
            Console.WriteLine("[Объединение] Не с чем объединять.");
            return CalculateTotal();
        }
        decimal total = CalculateTotal() + other.CalculateTotal();
        Console.WriteLine($"[Объединение] «{ProductName}» + «{other.ProductName}» = {total:C}");
        return total;
    }
}

//учёт кол-ва товара 
class StandardLine : OrderLine
{
    private decimal units; // денежный тип, чтобы легче умножить на цену

    public decimal Units
    {
        get { return units; }
        set
        {
            if (value <= 0)
                throw new ArgumentOutOfRangeException("Количество должно быть положительным!");
            units = value;
        }
    }

    public StandardLine() : base()
    {
        Units = 1m;
    }

    // Параметризованный — инициализация через сеттеры базового и своего
    public StandardLine(int productId, string productName, decimal price, decimal units)
        : base(productId, productName, price)
    {
        Units = units;
    }

    public override decimal CalculateTotal()
    {
        return Price * Units;
    }

    // доп метод для взаимодействия с другими объектами
    public void AddUnitsTo(StandardLine other, decimal extraUnits)
    {
        if (other == null)
        {
            Console.WriteLine("[StandardLine] Некому добавлять количество.");
            return;
        }
        other.Units += extraUnits;
        Console.WriteLine($"[StandardLine] «{ProductName}» добавляет {extraUnits} шт. товару «{other.ProductName}». Теперь: {other.Units} шт.");
    }
}

// добавляем скидки
class SpecialLine : OrderLine
{
    // Скидка в процентах
    private decimal discount;

    public decimal Discount
    {
        get { return discount; }
        set
        {
            if (value < 0 || value > 100)
                throw new ArgumentOutOfRangeException("Скидка должна быть от 0 до 100%!");
            discount = value;
        }
    }

    public SpecialLine() : base()
    {
        Discount = 0m;
    }

    public SpecialLine(int productId, string productName, decimal price, decimal discount)
        : base(productId, productName, price)
    {
        Discount = discount;
    }

    public override decimal CalculateTotal()
    {
        return Price * (1 - Discount / 100);
    }

    public new void UpdatePrice(decimal newPrice)
    {
        decimal discountedPrice = newPrice * (1 - Discount / 100);
        Price = discountedPrice;
        Console.WriteLine($"Цена со скидкой {Discount}%: {Price:C}");
    }

    // Взаимодействие -подарить часть своей скидки другому товару
    public void ShareDiscountWith(OrderLine other, decimal percentToShare)
    {
        if (other == null)
        {
            Console.WriteLine("[SpecialLine] Некому делиться скидкой.");
            return;
        }
        if (percentToShare < 0 || percentToShare > Discount)
        {
            Console.WriteLine($"[SpecialLine] Нельзя поделиться скидкой больше, чем у тебя ({Discount}%).");
            return;
        }
        decimal newPrice = other.Price * (1 - percentToShare / 100);
        other.Price = newPrice;
        Console.WriteLine($"[SpecialLine] «{ProductName}» делится скидкой {percentToShare}% с «{other.ProductName}». Новая цена: {other.Price:C}");
    }
}

// учёт предоплаты 
class FreeLine : OrderLine
{
    private decimal prepayment;

    public decimal Prepayment
    {
        get { return prepayment; }
        set
        {
            if (value < 0)
                throw new ArgumentOutOfRangeException("Предоплата не может быть отрицательной!");
            prepayment = value;
        }
    }

    public FreeLine() : base()
    {
        Prepayment = 0m;
    }

    public FreeLine(int productId, string productName, decimal price, decimal prepayment)
        : base(productId, productName, price)
    {
        Prepayment = prepayment;
    }

    // Если остаток отрицательный, возвращается 0 (товар полностью оплачен)
    public override decimal CalculateTotal()
    {
        decimal remaining = Price - Prepayment;
        // Тернарный оператор
        return remaining > 0 ? remaining : 0;
    }

    // взаимодействие - оплатить часть стоимости другого товара
    public void PayFor(OrderLine other, decimal amount)
    {
        if (other == null)
        {
            Console.WriteLine("[FreeLine] Некому оплачивать.");
            return;
        }
        if (amount <= 0 || amount > Prepayment)
        {
            Console.WriteLine($"[FreeLine] Некорректная сумма или недостаточно предоплаты ({Prepayment:C}).");
            return;
        }
        other.Price -= amount;
        Prepayment -= amount;
        Console.WriteLine($"[FreeLine] «{ProductName}» оплатил {amount:C} за «{other.ProductName}». Остаток предоплаты: {Prepayment:C}");
    }
}


Console.WriteLine("=== Создание объектов (через конструкторы с геттерами/сеттерами) ===\n");

StandardLine myStandard = new StandardLine(101, "Ноутбук", 50000m, 2);
SpecialLine mySpecial = new SpecialLine(102, "Телефон", 30000m, 15);
FreeLine myFree = new FreeLine(103, "Планшет", 25000m, 20000m);

OrderLine[] items = { myStandard, mySpecial, myFree };

Console.WriteLine("\n=== Все товары в заказе (полиморфизм) ===\n");
foreach (var item in items)
{
    Console.WriteLine($"  {item.GetProductDetails()}");
    Console.WriteLine($"  Итого: {item.CalculateTotal():C}\n");
}

Console.WriteLine("=== Специализированные методы производных классов ===\n");

Console.WriteLine("  SpecialLine — обновление цены со скидкой:");
mySpecial.UpdatePrice(35000m);

Console.WriteLine("\n  FreeLine — остаток после предоплаты:");
Console.WriteLine($"  Остаток к оплате: {myFree.CalculateTotal():C}\n");

//объкт базового класса OrderLine, без специальных/дополнительных полей
OrderLine basic = new OrderLine(999, "Обычный товар", 1000m);

Console.WriteLine("=== Базовый класс OrderLine (отдельный объект) ===\n");
Console.WriteLine($"  {basic.GetProductDetails()}");
Console.WriteLine($"  Итого: {basic.CalculateTotal():C}\n");


Console.WriteLine("=== Взаимодействие объектов между собой ===\n");

//Сравнение стоимости
myStandard.CompareWith(mySpecial);
Console.WriteLine();

//Скидка от SpecialLine на другой товар
mySpecial.ApplyDiscountTo(basic, 10);
Console.WriteLine();

//Объединение стоимости двух товаров
myStandard.CombineWith(mySpecial);
Console.WriteLine();

//tandardLine добавляет количество другому StandardLine
StandardLine anotherStandard = new StandardLine(104, "Монитор", 15000m, 1);
myStandard.AddUnitsTo(anotherStandard, 3);
Console.WriteLine($"  anotherStandard.Total = {anotherStandard.CalculateTotal():C}\n");

//SpecialLine делится скидкой с другим товаром
mySpecial.ShareDiscountWith(myFree, 5);
Console.WriteLine();

//FreeLine оплачивает часть стоимости другого товара из своей предоплаты
myFree.PayFor(myStandard, 5000m);
Console.WriteLine();

//Итоговое сравнение после всех взаимодействий
myStandard.CompareWith(myFree);
Console.WriteLine();

Console.WriteLine("=== Итоговое состояние объектов ===\n");
foreach (var item in items)
{
    Console.WriteLine($"  {item.GetProductDetails()}");
    Console.WriteLine($"  Итого: {item.CalculateTotal():C}\n");
}

The below script needs to be able to find the current output cell; this is an easy method to get it.

=== Создание объектов (через конструкторы с геттерами/сеттерами) ===


=== Все товары в заказе (полиморфизм) ===

  ID: 101, Название: Ноутбук, Цена: 50 000,00 ₽
  Итого: 100 000,00 ₽

  ID: 102, Название: Телефон, Цена: 30 000,00 ₽
  Итого: 25 500,00 ₽

  ID: 103, Название: Планшет, Цена: 25 000,00 ₽
  Итого: 5 000,00 ₽

=== Специализированные методы производных классов ===

  SpecialLine — обновление цены со скидкой:
Цена со скидкой 15%: 29 750,00 ₽

  FreeLine — остаток после предоплаты:
  Остаток к оплате: 5 000,00 ₽

=== Базовый класс OrderLine (отдельный объект) ===

  ID: 999, Название: Обычный товар, Цена: 1 000,00 ₽
  Итого: 1 000,00 ₽

=== Взаимодействие объектов между собой ===

[Сравнение] «Ноутбук» (100 000,00 ₽) vs «Телефон» (25 287,50 ₽)
  → «Ноутбук» дороже на 74 712,50 ₽

[Скидка] «Телефон» дарит скидку 10% товару «Обычный товар». Новая цена: 900,00 ₽

[Объединение] «Ноутбук» + «Телефон» = 125 287,50 ₽

[StandardLine] «Ноутбук» добавляет 3 шт. товару «Монитор». Теперь: